Bilkul. Neeche **copy-paste ready notes** hain. Tum isko apne HITL code ke **upar paste** kar sakte ho. Maine sequence mein rakha hai: **kya hai → kyun → kaha lagta hai → code mein kya karta hai**.

# LangGraph HITL (Human-in-the-Loop) — Quick Notes

## 1. HITL kya hai?

HITL = **Human-in-the-Loop**

Iska matlab:

> Agent/Graph kisi important step par rukta hai aur human se input ya approval leta hai, phir human ke response ke according continue karta hai.

Basic flow:

User
↓
Graph / Agent
↓
Important action
↓
`interrupt()` 🛑
↓
Human input / approval
↓
`Command(resume=...)`
↓
Graph continue
↓
Final result


## 2. HITL kyun use karte hain?

Jab agent ko kisi important ya risky action se pehle human ki permission chahiye.

Examples:

- Payment karna
- Email send karna
- Booking karna
- Property viewing schedule karna
- Data delete karna
- Important information change karna

Normal/simple kaam ke liye HITL zaroori nahi.

Example:

Search property
→ Agent khud kare

Property details show
→ Agent khud kare

Availability check
→ Agent khud kare

Viewing book karna
→ Human approval le sakte hain


## 3. `interrupt()` kya karta hai?

`interrupt()` graph ko **pause** karta hai.

Example:

```python
decision = interrupt({
    "type": "approval",
    "question": "Do you want to book this viewing?",
    "instruction": "Approve or reject"
})
```

Yahan graph ruk jayega.

Important:

`interrupt()` human ko automatically Python terminal mein input nahi karwata.

Ye sirf graph ko pause karta hai aur interrupt data return karta hai.

Frontend / terminal / application ko human se response lena hota hai.


## 4. `interrupt()` kahan lagta hai?

Rule:

> Jis node mein human ki zaroorat hai, us node ke andar `interrupt()` lagta hai.

Example:

```text
search_node
    ↓
availability_node
    ↓
booking_node
    ↓
interrupt() 🛑
    ↓
booking continue
```


## 6. `interrupt()` ke andar kya dete hain?

Hum ek dictionary dete hain:

```python
interrupt({
    "type": "approval",
    "question": "Do you want to continue?",
    "instruction": "Approve or reject"
})
```

Ye information human/frontend ko batane ke liye hoti hai.

### `type`

Batata hai interrupt kis type ka hai.

Example:

```python
"type": "approval"
```

Matlab human se approval chahiye.

Ya:

```python
"type": "input"
```

Matlab human se information leni hai.


### `question`

Human ko kya poochna hai.

Example:

```python
"question": "Do you want to book this property?"
```


### `instruction`

Human ko kya karna hai.

Example:

```python
"instruction": "Approve or reject"
```


## 8. Checkpointer kyun chahiye?

Interrupt ke waqt graph ruk raha hai.

Graph ko apni state yaad rakhni hoti hai taake baad mein wahi se continue kar sake.

Isliye checkpointer use hota hai:

```python
from langgraph.checkpoint.memory import MemorySaver

checkpointer = MemorySaver()
```

Phir:

```python
app = builder.compile(
    checkpointer=checkpointer
)
```

Simple meaning:

> Checkpointer = graph ki state ko save/remember karne wala system.


## 9. `thread_id` kyun chahiye?

Graph ko pata hona chahiye ke **kis conversation ko resume karna hai**.

Example:

```python
config = {
    "configurable": {
        "thread_id": "1234"
    }
}
```

Think:

```text
Thread 1234
    ↓
User conversation
    ↓
interrupt()
    ↓
PAUSE
    ↓
resume
    ↓
same Thread 1234
```

Agar multiple users/conversations hain to har conversation ka alag `thread_id` ho sakta hai.


## 10. First `invoke()` kya karta hai?

Example:

```python
result = app.invoke(
    initial_input,
    config=config
)
```

Graph run hota hai.

Jab graph `interrupt()` par pahunchta hai:

```text
Graph
 ↓
interrupt()
 ↓
PAUSE
```

Aur `result` mein interrupt information milti hai.


## 11. `__interrupt__` kya hai?

LangGraph interrupt hone par result mein special key deta hai:

```python
result["__interrupt__"]
```

Ye interrupt information contain karti hai.

Example:

```python
result["__interrupt__"]
```

roughly:

```python
[
    Interrupt(...)
]
```


## 12. `[0]` kyun?

`__interrupt__` ek list hoti hai.

```python
result["__interrupt__"][0]
```

Matlab:

> List ka first interrupt object lo.


## 13. `.value` kyun?

Interrupt object ke andar humara actual data `value` mein hota hai.

Isliye:

```python
message = result["__interrupt__"][0].value
```

Meaning:

> LangGraph ke interrupt object ke andar se actual interrupt data nikaal kar `message` mein save karo.


## 14. Human se input lena

Example terminal application:

```python
user_input = input(
    f"{message}\nApprove? (y/n): "
)
```

Ab human answer deta hai:

```text
y
```

ya:

```text
n
```


## 15. `Command(resume=...)` kya karta hai?

Human response milne ke baad graph ko resume karna hota hai.

Iske liye:

```python
Command(resume=...)
```

use hota hai.

Example:

```python
app.invoke(
    Command(
        resume={"approved": user_input}
    ),
    config=config
)
```

Simple meaning:

> "Jahan graph interrupt hua tha, wahi se continue karo aur ye human response use karo."


## 16. Same `thread_id` important hai

First invoke:

```python
app.invoke(
    initial_input,
    config=config
)
```

Resume:

```python
app.invoke(
    Command(resume={"approved": user_input}),
    config=config
)
```

Dono mein **same `config` / same `thread_id`** hona chahiye.

```text
thread 1234
    ↓
interrupt
    ↓
human response
    ↓
thread 1234
    ↓
resume
```


## 17. Complete HITL sequence

### Step 1 — Import

```python
from langgraph.types import interrupt, Command
```

`interrupt` = graph pause

`Command` = graph resume


### Step 2 — State

```python
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
```


### Step 3 — Node

Jahan human ki zaroorat hai:

```python
def booking_node(state):

    decision = interrupt({
        "type": "approval",
        "question": "Do you want to book this viewing?",
        "instruction": "Approve or reject"
    })

    if decision["approved"]:
        return {
            "messages": [
                AIMessage(content="Booking confirmed.")
            ]
        }

    return {
        "messages": [
            AIMessage(content="Booking cancelled.")
        ]
    }
```


### Step 4 — Checkpointer

```python
checkpointer = MemorySaver()
```


### Step 5 — Compile with checkpointer

```python
app = builder.compile(
    checkpointer=checkpointer
)
```


### Step 6 — Thread ID

```python
config = {
    "configurable": {
        "thread_id": "1234"
    }
}
```


### Step 7 — First invoke

```python
result = app.invoke(
    initial_input,
    config=config
)
```

Graph `interrupt()` par rukega.


### Step 8 — Interrupt data nikalo

```python
message = result["__interrupt__"][0].value
```


### Step 9 — Human se response lo

```python
user_input = input(
    f"{message}\nApprove? (y/n): "
)
```


### Step 10 — Resume

```python
final_result = app.invoke(
    Command(
        resume={
            "approved": user_input
        }
    ),
    config=config
)
```


## 18. Complete mental picture

```text
                USER
                  ↓
             Graph starts
                  ↓
              Node runs
                  ↓
        Important action?
             ↙        ↘
           NO          YES
           ↓            ↓
       Continue      interrupt()
                        ↓
                     PAUSE 🛑
                        ↓
                 Save graph state
                        ↓
                 Human sees question
                        ↓
                  Human responds
                        ↓
              Command(resume=...)
                        ↓
                 Same thread_id
                        ↓
                Graph continues
                        ↓
                  Final result
```


# 19. Golden Rule

Yaad rakhne ke liye sirf ye line:

> **`interrupt()` = PAUSE**
>
> **Human input = DECISION**
>
> **`Command(resume=...)` = CONTINUE**
>
> **Checkpointer = STATE SAVE**
>
> **`thread_id` = WHICH CONVERSATION**



# 6. First run
result = app.invoke(
    initial_input,
    config=config
)


# 7. Interrupt data
message = result["__interrupt__"][0].value


# 8. Human input
user_input = input(
    f"{message}\nApprove? (y/n): "
)


# 9. Resume graph
final_result = app.invoke(
    Command(
        resume={
            "approved": user_input
        }
    ),
    config=config
)
```

## ONE-LINE REVISION

```text
interrupt() → graph rokta hai
↓
__interrupt__ → interrupt ka returned data
↓
.value → humara actual interrupt payload
↓
Human input → user ka decision
↓
Command(resume=...) → graph ko continue karta hai
↓
checkpointer → state save rakhta hai
↓
thread_id → correct conversation identify karta hai
```

Isko code ke upar rakhne ke liye **ye format deliberately notes/reference style mein hai**, taake baad mein code dekhte waqt turant samajh aa jaye ke har HITL cheez kya kar rahi hai.

In [28]:
from typing import Annotated

from langchain_core.messages import AnyMessage, AIMessage
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage

In [29]:
load_dotenv()
model=ChatGroq(model='openai/gpt-oss-120b')

In [30]:
class state(TypedDict):
    messages:Annotated[list, add_messages]
    

In [ ]:
#Interrupt ka istemal: Tool ke code mein interrupt function ko call kiya jata hai (36:38). Yeh function wahi message (jaise: "Approve buying 10 shares of Apple?") prepare karta hai jo user ko dikhana hota hai 
def chat_node(state:state)->state:
    decision = interrupt({
        "type": "approval",
        "reason": "Model is about to answer a user question.",
        "question": state["messages"][-1].content,
        "instruction": "Approve this question? yes/no"
    })
    
    if decision["approved"] == 'no':
        return {"messages": [AIMessage(content="Not approved.")]}

    else:
        response = model.invoke(state["messages"])
        return {"messages": [response]}

In [32]:
builder=StateGraph(state)

builder.add_node('chat_node', chat_node)

builder.add_edge(START, 'chat_node')
builder.add_edge('chat_node', END)

checkpointer= MemorySaver()
flow=builder.compile(checkpointer=checkpointer)

In [33]:
config={'configurable':{'thread_id':'123'}}

intial_state={'messages':[
    ('user', "exlain python in points")
]}
result = flow.invoke(intial_state, config=config)

In [34]:
result

{'messages': [HumanMessage(content='exlain python in points', additional_kwargs={}, response_metadata={}, id='99f62212-2fb3-4eab-abdf-2d5bae730af3')],
 '__interrupt__': [Interrupt(value={'type': 'approval', 'reason': 'Model is about to answer a user question.', 'question': 'exlain python in points', 'instruction': 'Approve this question? yes/no'}, id='5c30521ef974b24ed7f3be103b8bf296')]}

In [35]:
message= result['__interrupt__'][0].value
message

{'type': 'approval',
 'reason': 'Model is about to answer a user question.',
 'question': 'exlain python in points',
 'instruction': 'Approve this question? yes/no'}

In [ ]:
user= input(f"approve this {message} yes or not") 

In [37]:
final_result= flow.invoke(Command(resume={'approved':user}), config=config)
print(final_result['messages'][-1].content)
# Resume the graph with the approval decision


Not approved.
